# FlyPose-SAR — Fine-tuning YOLO Fase 2 (Dominio Termico)

## Prerequisiti (Data):
- `dataset_sar` — dataset RGB Fase 1 con label YOLO-pose (box + 17 kpts)
- `dataset_sar_thermal` — dataset sintetico GAN (stesse label, frame convertiti)
- `hit-uav` — termico reale zenitale (solo box, kpts assenti)
- `flypose-fase1-weights` — pesi Large SAR Fase 1 (`best_large_sar.pt`)

## Setup: GPU T4 x1 basta (o T4 x2 se disponibile)

## Cella 1 — Installazione

In [ ]:
!pip install ultralytics -q
import torch
from pathlib import Path
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## Cella 2 — Trova dataset e pesi

In [ ]:
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# ---- dataset_sar_thermal (sintetico GAN) ----
thermal_path = None
for d in KAGGLE_INPUT.rglob('dataset_sar_thermal'):
    if d.is_dir():
        thermal_path = d
        break

# ---- HIT-UAV (termico reale) ----
hituav_path = None
for d in KAGGLE_INPUT.rglob('hit-uav'):
    if d.is_dir():
        hituav_path = d
        break
if hituav_path is None:
    for d in KAGGLE_INPUT.rglob('images'):
        root = d.parent
        if 'hit' in str(root).lower() and (root / 'labels' / 'train').exists():
            hituav_path = root
            break

# ---- pesi Fase 1 ----
fase1_weights = None
for p in KAGGLE_INPUT.rglob('*.pt'):
    if 'large' in p.name.lower() or 'best' in p.name.lower():
        fase1_weights = p
        break
if fase1_weights is None:
    for p in KAGGLE_INPUT.rglob('*.pt'):
        fase1_weights = p
        break

print(f'Sintetico GAN  : {thermal_path}')
print(f'HIT-UAV reale  : {hituav_path}')
print(f'Pesi Fase 1    : {fase1_weights}')

if any(x is None for x in [thermal_path, hituav_path, fase1_weights]):
    print('\nERRORE: uno o più dataset mancanti — controlla i path sopra')

## Cella 3 — Costruisce il dataset misto

In [ ]:
import shutil, random
from pathlib import Path

# ---- PARAMETRI ----
SUBSAMPLE_SYNTHETIC = 4   # prende 1 frame ogni N dal sintetico (riduce ridondanza video)
HITUAV_PERSON_CLASS = 0   # classe Person in HIT-UAV
N_KPT               = 17  # keypoints COCO
VAL_SPLIT           = 0.1 # 10% del sintetico come val
SEED                = 42
# -------------------

random.seed(SEED)
mix_root = KAGGLE_WORKING / 'dataset_thermal_mix'
for split in ['train', 'val']:
    (mix_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (mix_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

def copy_pair(src_img, src_lbl, dst_img_dir, dst_lbl_dir, stem_prefix=''):
    """Copia immagine e label con prefisso opzionale per evitare collisioni di nome."""
    dst_stem = stem_prefix + src_img.stem
    shutil.copy(src_img, dst_img_dir / (dst_stem + src_img.suffix))
    if src_lbl.exists():
        shutil.copy(src_lbl, dst_lbl_dir / (dst_stem + '.txt'))

# ---------- 1. Sintetico GAN (subsampled, train+val split) ----------
syn_imgs = sorted((thermal_path / 'images' / 'train').glob('*.*'))
syn_imgs = syn_imgs[::SUBSAMPLE_SYNTHETIC]  # 1 ogni N
random.shuffle(syn_imgs)
n_val = int(len(syn_imgs) * VAL_SPLIT)
syn_val, syn_train = syn_imgs[:n_val], syn_imgs[n_val:]

syn_lbl_dir = thermal_path / 'labels' / 'train'
for p in syn_train:
    copy_pair(p, syn_lbl_dir / (p.stem + '.txt'),
              mix_root / 'images' / 'train',
              mix_root / 'labels' / 'train', stem_prefix='syn_')
for p in syn_val:
    copy_pair(p, syn_lbl_dir / (p.stem + '.txt'),
              mix_root / 'images' / 'val',
              mix_root / 'labels' / 'val', stem_prefix='syn_')

print(f'Sintetico train: {len(syn_train)} | val: {len(syn_val)}')

# ---------- 2. HIT-UAV reale: filtra solo Person, aggiunge kpts=0 ----------
hit_img_dir = hituav_path / 'images' / 'train'
hit_lbl_dir = hituav_path / 'labels' / 'train'
kpt_zeros   = ' '.join(['0 0 0'] * N_KPT)  # 17 keypoints tutti non visibili

n_hit = 0
for lbl_path in sorted(hit_lbl_dir.glob('*.txt')):
    lines_in  = lbl_path.read_text().strip().splitlines()
    lines_out = []
    for line in lines_in:
        parts = line.split()
        if not parts:
            continue
        if int(float(parts[0])) != HITUAV_PERSON_CLASS:
            continue  # scarta non-persone
        # rimappa a classe 0 e appende 17 kpts a visibilità 0
        lines_out.append('0 ' + ' '.join(parts[1:5]) + ' ' + kpt_zeros)
    if not lines_out:
        continue  # skip immagini senza persone

    # cerca l'immagine corrispondente
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = hit_img_dir / (lbl_path.stem + ext)
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        continue

    dst_stem = 'hit_' + lbl_path.stem
    shutil.copy(img_path, mix_root / 'images' / 'train' / (dst_stem + img_path.suffix))
    (mix_root / 'labels' / 'train' / (dst_stem + '.txt')).write_text('\n'.join(lines_out))
    n_hit += 1

print(f'HIT-UAV reale  : {n_hit} immagini con persone')

# ---------- 3. Riepilogo ----------
n_train_tot = len(list((mix_root / 'images' / 'train').glob('*.*')))
n_val_tot   = len(list((mix_root / 'images' / 'val').glob('*.*')))
print(f'\nDataset misto finale:')
print(f'  Train: {n_train_tot} immagini ({len(syn_train)} sintetiche + {n_hit} reali HIT-UAV)')
print(f'  Val  : {n_val_tot} immagini (sintetiche)')

## Cella 4 — Scrive data.yaml

In [ ]:
import yaml

data_yaml = {
    'path'      : str(mix_root),
    'train'     : 'images/train',
    'val'       : 'images/val',
    'nc'        : 1,
    'names'     : ['person'],
    'kpt_shape' : [17, 3],
}

yaml_path = mix_root / 'data_thermal.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'[OK] {yaml_path}')
print(open(yaml_path).read())

## Cella 5 — Fine-tuning YOLO11-Pose (Nano / Small / Large)

In [ ]:
from ultralytics import YOLO

# ---- PARAMETRI ----
MODEL_SIZE = 'large'   # 'nano' | 'small' | 'large'
EPOCHS     = 50
BATCH      = 16
IMGSZ      = 640
LR0        = 0.0002    # ~1/10 del lr Fase 1
LRF        = 0.01      # lr finale = LR0 * LRF
# -------------------

# mappa nome → pesi Fase 1
# se hai tutti e tre i modelli Fase 1, metti i path qui
weights_map = {
    'nano' : fase1_weights,  # sostituisci con path specifico se hai più .pt
    'small': fase1_weights,
    'large': fase1_weights,
}

model = YOLO(str(weights_map[MODEL_SIZE]))

results = model.train(
    data      = str(yaml_path),
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMGSZ,
    lr0       = LR0,
    lrf       = LRF,
    warmup_epochs = 3,
    cos_lr    = True,
    name      = f'flypose_thermal_{MODEL_SIZE}',
    project   = str(KAGGLE_WORKING / 'runs_thermal'),
    exist_ok  = True,
    device    = 0,
    workers   = 4,
    patience  = 20,   # early stopping se non migliora per 20 epoche
    save      = True,
    plots     = True,
)

print(f'\n[OK] Training completato: {MODEL_SIZE}')
print(f'Best weights: {results.save_dir}/weights/best.pt')

## Cella 6 — Valutazione formale sul val set

In [ ]:
from ultralytics import YOLO

best_pt = list((KAGGLE_WORKING / 'runs_thermal' / f'flypose_thermal_{MODEL_SIZE}' / 'weights').glob('best.pt'))[0]
model   = YOLO(str(best_pt))

metrics = model.val(
    data    = str(yaml_path),
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = 0,
)

print(f'\n=== Risultati Fase 2 — {MODEL_SIZE.upper()} SAR Thermal ===')
print(f'Box  mAP@0.5     : {metrics.box.map50:.4f}')
print(f'Pose mAP@0.5     : {metrics.pose.map50:.4f}')
print(f'Box  Precision   : {metrics.box.mp:.4f}')
print(f'Box  Recall      : {metrics.box.mr:.4f}')
print(f'\n(confronto Fase 1 Large SAR RGB: Box mAP50=0.5961, Pose mAP50=0.3852)')

## Cella 7 — Salva zip con pesi e risultati

In [ ]:
import zipfile

zip_path = KAGGLE_WORKING / f'flypose_thermal_{MODEL_SIZE}.zip'
run_dir  = KAGGLE_WORKING / 'runs_thermal' / f'flypose_thermal_{MODEL_SIZE}'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pt in (run_dir / 'weights').glob('*.pt'):
        zf.write(pt, f'weights/{pt.name}')
        print(f'  [+] {pt.name}')
    for img in run_dir.glob('*.png'):
        zf.write(img, f'plots/{img.name}')
        print(f'  [+] {img.name}')

print(f'\n[OK] {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)')
print('Scarica da Output (pannello destro)')